# Cache Performance Comparison: ReflectionCache vs SerializationCache

Compares O(1) verification results from:
- `ReflectionCache` (`spark.kotlin.reflect`) — keyed by `KType`
- `SerializationCache` (`spark.kotlin.serialization`) — keyed by `SerialDescriptor`

**Test suite:** 5 microbenchmark tests, same methodology for both caches.
**Sources:** `test-results/performance-reflect/`, `test-results/performance-serialization/`

In [1]:
%use lets-plot

In [2]:
import java.io.File

// Handles quoted fields — locale decimal comma inside quotes is preserved.
fun parseCsvLine(line: String): List<String> {
    val result = mutableListOf<String>()
    var inQuotes = false
    val current = StringBuilder()
    for (c in line) {
        when {
            c == '"' -> inQuotes = !inQuotes
            c == ',' && !inQuotes -> {
                result.add(current.toString()); current.clear()
            }

            else -> current.append(c)
        }
    }
    result.add(current.toString())
    return result
}

// Known column layout for PerformanceDataWriter output (metrics sorted alphabetically).
val BASE_COLS = listOf(
    "test_name", "test_id", "execution_id", "execution_date", "execution_time",
    "sample_size", "run_number", "time_ns", "warmup_iterations",
    "jvm_version", "jvm_vendor", "os_name", "os_version", "os_arch",
    "cpu_cores", "max_memory_mb", "total_memory_mb", "free_memory_mb",
    "kotlin_version", "gradle_version", "test_result", "test_status", "failure_reason"
)
val KNOWN_HEADERS = mapOf(
    "test1_measurement_validation.csv" to BASE_COLS +
            listOf("test1_data_structure", "test1_expected_complexity", "test1_structure_growth_10k_100k"),
    "test2_constant_time_growth.csv" to BASE_COLS +
            listOf(
                "test2_p10_ns", "test2_p50_ns", "test2_p50_threshold_ns", "test2_p90_ns", "test2_p99_ns",
                "test2_spread_p90_p10", "test2_spread_threshold"
            ),
    "test3_linear_regression.csv" to BASE_COLS +
            listOf("test3_r_squared", "test3_regression_intercept", "test3_regression_slope", "test3_slope_threshold"),
    "test4_memory_pressure.csv" to BASE_COLS +
            listOf(
                "test4_baseline_time_ns", "test4_degradation_ratio", "test4_degradation_threshold",
                "test4_memory_pressure_mb", "test4_pressure_time_ns"
            ),
    "test5_type_complexity.csv" to BASE_COLS +
            listOf("test5_field_count", "test5_type_complexity", "test5_variance_ratio", "test5_variance_threshold")
)

// Returns list of maps: column -> value.
// Skips # comment lines. Uses embedded header row when present (starts with "test_name"),
// otherwise falls back to KNOWN_HEADERS by filename.
fun loadCsv(file: File): List<Map<String, String>> {
    val lines = file.readLines().filter { !it.startsWith("#") && it.isNotBlank() }
    if (lines.isEmpty()) return emptyList()
    val hasHeader = lines.first().startsWith("test_name")
    val headers: List<String>
    val dataLines: List<String>
    if (hasHeader) {
        headers = parseCsvLine(lines.first())
        dataLines = lines.drop(1)
    } else {
        headers = KNOWN_HEADERS[file.name] ?: parseCsvLine(lines.first()).also { return emptyList() }
        dataLines = lines
    }
    return dataLines.map { line ->
        val values = parseCsvLine(line)
        headers.zip(values).toMap()
    }
}

// Parse a possibly locale-formatted double ("3,14" → 3.14).
fun String.toDoubleL(): Double = this.replace(',', '.').toDouble()
fun String.toLongL(): Long = this.replace(',', '.').toDoubleOrNull()?.toLong() ?: this.toLong()

val reflectDir = File("../test-results/performance-reflect")
val serializeDir = File("../test-results/performance-serialization")

fun latestRunRows(rows: List<Map<String, String>>): List<Map<String, String>> {
    if (rows.isEmpty()) return rows
    val ids = rows.mapNotNull { it["execution_id"]?.takeIf { id -> id.isNotBlank() } }
    if (ids.isEmpty()) return rows
    val latestId = ids.last()
    return rows.filter { it["execution_id"] == latestId }
}

fun load(dir: File, testFile: String, cache: String): List<Map<String, String>> =
    latestRunRows(loadCsv(File(dir, testFile))).map { it + ("cache" to cache) }

println("reflect dir exists:   ${reflectDir.exists()}")
println("serialize dir exists: ${serializeDir.exists()}")
println("reflect files:   ${reflectDir.listFiles()?.map { it.name }}")
println("serialize files: ${serializeDir.listFiles()?.map { it.name }}")

reflect dir exists:   true
serialize dir exists: true
reflect files:   [test3_linear_regression.csv, test1_measurement_validation.csv, test4_memory_pressure.csv, test2_constant_time_growth.csv, test5_type_complexity.csv]
serialize files: [test3_linear_regression.csv, test1_measurement_validation.csv, test4_memory_pressure.csv, test2_constant_time_growth.csv, test5_type_complexity.csv]


## 1. Measurement Validation

Confirms the benchmarking method distinguishes O(1) from O(n).
Both caches are compared against the same HashMap reference lookup.
Expected: cache lookup time is comparable to HashMap, far below ArrayList.

In [3]:
val t1Reflect = load(reflectDir, "test1_measurement_validation.csv", "reflect")
val t1Serialize = load(serializeDir, "test1_measurement_validation.csv", "serialize")

fun t1CacheName(cache: String) = if (cache == "reflect") "ReflectionCache" else "SerializationCache"

data class T1Point(val label: String, val timeNs: Long)

// One HashMap reference (from reflect file), then one row per cache.
val hashMapRow = t1Reflect.firstOrNull { it["test1_data_structure"] == "HashMap" && it["sample_size"] == "10000" }
    ?: t1Reflect.firstOrNull { it["test1_data_structure"] == "HashMap" }

val t1Points = buildList {
    if (hashMapRow != null)
        add(T1Point("HashMap\n(reference)", hashMapRow["time_ns"]!!.toLongL()))
    listOf("reflect" to t1Reflect, "serialize" to t1Serialize).forEach { (cache, rows) ->
        rows.firstOrNull { it["test1_data_structure"] == t1CacheName(cache) }
            ?.let { add(T1Point("${t1CacheName(cache)}\n($cache)", it["time_ns"]!!.toLongL())) }
    }
}

println("Test 1 — cache lookup time vs HashMap (10k ops):")
t1Points.forEach { println("  %-35s  %,d ns".format(it.label.replace("\n", " "), it.timeNs)) }

val t1Data = mapOf(
    "label" to t1Points.map { it.label },
    "time_ns" to t1Points.map { it.timeNs.toDouble() }
)
val t1Order = listOf("HashMap\n(reference)", "ReflectionCache\n(reflect)", "SerializationCache\n(serialize)")

letsPlot(t1Data) +
        geomBar(stat = Stat.identity, alpha = 0.85) { x = "label"; y = "time_ns"; fill = "label" } +
        scaleXDiscrete(limits = t1Order) +
        labs(
            title = "Cache lookup time vs HashMap reference  (10k ops per sample, ns)",
            x = "Structure", y = "Time per op (ns)", fill = "Structure"
        ) +
        ggsize(680, 380)

Test 1 — cache lookup time vs HashMap (10k ops):
  HashMap (reference)                  76 ns
  ReflectionCache (reflect)            41 ns
  SerializationCache (serialize)       144 ns


HashMap 
 
 
 (reference) 
 
 
 
 
 
 
 
 
 ReflectionCache 
 
 
 (reflect) 
 
 
 
 
 
 
 
 
 SerializationCache 
 
 
 (serialize) 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 120 
 
 
 
 
 
 
 140 
 
 
 
 
 
 
 
 
 Cache lookup time vs HashMap reference (10k ops per sample, ns) 
 
 
 
 
 Time per op (ns) 
 
 
 
 
 Structure 
 
 
 
 
 
 
 
 
 Structure 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 HashMap 
 
 
 (reference) 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 ReflectionCache 
 
 
 (reflect) 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 SerializationCache 
 
 
 (serialize)

## 2. Lookup Stability (P90/P10 Spread)

50 independent samples × 10 000 ops each. Percentiles computed across all samples.
A low spread confirms constant time — no per-call accumulation or recomputation.
Threshold: P90/P10 < 20×, median < 10 µs.

In [4]:
val t2Reflect = load(reflectDir, "test2_constant_time_growth.csv", "reflect")
val t2Serialize = load(serializeDir, "test2_constant_time_growth.csv", "serialize")

// Metric columns are identical on every row — just read from first row.
data class T2Summary(val cache: String, val p10: Long, val p50: Long, val p90: Long, val p99: Long, val spread: Double)

fun t2Summary(rows: List<Map<String, String>>, cache: String): T2Summary {
    val r = rows.first()
    return T2Summary(
        cache = cache,
        p10 = r["test2_p10_ns"]!!.toLongL(),
        p50 = r["test2_p50_ns"]!!.toLongL(),
        p90 = r["test2_p90_ns"]!!.toLongL(),
        p99 = r["test2_p99_ns"]!!.toLongL(),
        spread = r["test2_spread_p90_p10"]!!.toDoubleL()
    )
}

val t2R = t2Summary(t2Reflect, "reflect")
val t2S = t2Summary(t2Serialize, "serialize")

println("Test 2 — Lookup Stability:")
println(
    "  %-12s  p10=%,4d ns  p50=%,4d ns  p90=%,4d ns  p99=%,4d ns  spread=%.2fx"
        .format("reflect", t2R.p10, t2R.p50, t2R.p90, t2R.p99, t2R.spread)
)
println(
    "  %-12s  p10=%,4d ns  p50=%,4d ns  p90=%,4d ns  p99=%,4d ns  spread=%.2fx"
        .format("serialize", t2S.p10, t2S.p50, t2S.p90, t2S.p99, t2S.spread)
)

// Grouped bar: p10 / p50 / p90 per cache
val t2Caches = listOf("reflect", "reflect", "reflect", "serialize", "serialize", "serialize")
val t2Pcts = listOf("p10", "p50", "p90", "p10", "p50", "p90")
val t2Times = listOf(t2R.p10, t2R.p50, t2R.p90, t2S.p10, t2S.p50, t2S.p90).map { it.toDouble() }

val t2Data = mapOf("cache" to t2Caches, "percentile" to t2Pcts, "time_ns" to t2Times)

letsPlot(t2Data) +
        geomBar(stat = Stat.identity, position = positionDodge(0.9), alpha = 0.85) {
            x = "percentile"; y = "time_ns"; fill = "cache"
        } +
        scaleXDiscrete(limits = listOf("p10", "p50", "p90")) +
        labs(
            title = "Lookup latency percentiles by cache  (ns per op)",
            x = "Percentile", y = "Latency (ns)", fill = "Cache"
        ) +
        ggsize(650, 380)

Test 2 — Lookup Stability:
  reflect       p10=  34 ns  p50=  35 ns  p90=  38 ns  p99=  42 ns  spread=1,12x
  serialize     p10=  33 ns  p50=  34 ns  p90=  37 ns  p99=  40 ns  spread=1,12x


p10 
 
 
 
 
 
 
 
 
 p50 
 
 
 
 
 
 
 
 
 p90 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 
 
 Lookup latency percentiles by cache (ns per op) 
 
 
 
 
 Latency (ns) 
 
 
 
 
 Percentile 
 
 
 
 
 
 
 
 
 Cache 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 reflect 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 serialize

## 3. Linear Regression (Slope Near Zero)

Time per operation measured across batch sizes 10 000 → 200 000.
A slope near zero confirms O(1): time does not grow with call count.
Threshold: |slope| < 0.1 ns/op.

In [5]:
val t3Reflect = load(reflectDir, "test3_linear_regression.csv", "reflect")
val t3Serialize = load(serializeDir, "test3_linear_regression.csv", "serialize")

data class T3Summary(val cache: String, val slope: Double, val intercept: Double, val rSquared: Double)

fun t3Summary(rows: List<Map<String, String>>, cache: String): T3Summary {
    val r = rows.first()
    return T3Summary(
        cache = cache,
        slope = r["test3_regression_slope"]!!.toDoubleL(),
        intercept = r["test3_regression_intercept"]!!.toDoubleL(),
        rSquared = r["test3_r_squared"]!!.toDoubleL()
    )
}

val t3R = t3Summary(t3Reflect, "reflect")
val t3S = t3Summary(t3Serialize, "serialize")

println("Test 3 — Linear Regression:")
println("  reflect:   slope=%.6f ns/op  intercept=%.2f  R²=%.4f".format(t3R.slope, t3R.intercept, t3R.rSquared))
println("  serialize: slope=%.6f ns/op  intercept=%.2f  R²=%.4f".format(t3S.slope, t3S.intercept, t3S.rSquared))
println("\n  Both |slope| < 0.1 ns/op: ${Math.abs(t3R.slope) < 0.1 && Math.abs(t3S.slope) < 0.1}")

// Per-sample time_ns across batch sizes — scatter to show stability
val t3AllRows = t3Reflect.map { it + ("cache" to "reflect") } +
        t3Serialize.map { it + ("cache" to "serialize") }

val t3Data = mapOf(
    "cache" to t3AllRows.map { it["cache"]!! },
    "sample_size" to t3AllRows.map { it["sample_size"]!!.toDoubleL() },
    "time_ns" to t3AllRows.map { it["time_ns"]!!.toDoubleL() }
)

letsPlot(t3Data) +
        geomPoint(alpha = 0.35, size = 1.5) { x = "sample_size"; y = "time_ns"; color = "cache" } +
        geomSmooth(method = "lm", size = 1.0) { x = "sample_size"; y = "time_ns"; color = "cache" } +
        labs(
            title = "Time per op vs batch size (linear fit — slope ≈ 0 confirms O(1))",
            x = "Batch size (ops)", y = "Time per op (ns)", color = "Cache"
        ) +
        ggsize(700, 400)

Test 3 — Linear Regression:
  reflect:   slope=-0,000139 ns/op  intercept=29,66  R²=0,5913
  serialize: slope=-0,000349 ns/op  intercept=60,09  R²=0,4893

  Both |slope| < 0.1 ns/op: true


<path d="M24.994131472324558 204.32760896153346 L24.994131472324558 204.32760896153346 L31.32175969316622 205.2552219098911 L37.64938791400788 206.18193890739067 L43.97701613484954 207.10770790063634 L50.304644355691195 208.03247317483152 L56.632272576532856 208.95617507728815 L62.95990079737452 209.87874972277717 L69.28752901821619 210.80012868065134 L75.61515723905785 211.7202386440465 L81.9427854598995 212.6390010819606 L88.27041368074116 213.556331875646 L94.59804190158283 214.47214094155848 L100.92567012242449 215.3863318441176 L107.25329834326614 216.29880140278476 L113.58092656410781 217.20943929948774 L119.90855478494947 218.11812769425364 L126.23618300579112 219.02474085908057 L132.5638112266328 219.92914484260174 L138.89143944747443 220.8311971809769 L145.21906766831611 221.73074667365395 L151.54669588915777 222.62763324611882 L157.87432410999943 223.52168792536563 L164.2019523308411 224.4127329573905 L170.52958055168276 225.30058209925502 L176.85720877252442 226.18504112080768 L183.18483699336608 227.06590855250303 L189.5124652142077 227.94297671533224 L195.8400934350494 228.816033066006 L202.16772165589106 229.6848618845173 L208.4953498767327 230.54924632141925 L214.82297809757438 231.4089708081243 L221.150606318416 232.2638238151306 L227.4782345392577 233.1136009206722 L233.80586276009936 233.95810812687245 L240.133490980941 234.79716533378058 L246.46111920178268 235.6306098561416 L252.7887474226243 236.4582998463666 L259.116375643466 237.28011747313707 L265.4440038643076 238.09597170129766 L271.77163208514924 238.9058005272382 L278.0992603059909 239.7095725455333 L284.4268885268326 240.5072877561829 L290.75451674767424 241.29897756461247 L297.08214496851593 242.0847039744322 L303.4097731893576 242.86455802079738 L309.73740141019925 243.6386575350265 L316.0650296310409 244.40714436470853 L322.3926578518825 245.17018119509845 L328.7202860727242 245.92794812614702 L335.0479142935659 246.68063915573092 L341.3755425144075 247.428458705616 L347.7031707352492 248.1716183053042 L354.03079895609085 248.9103335233831 L360.35842717693254 249.6448212092995 L366.68605539777417 250.37529708306045 L373.0136836186158 251.10197368795528 L379.34131183945755 251.82505870299278 L385.6689400602992 252.5447535977184 L391.9965682811408 253.26125260228372 L398.32419650198244 253.97474195962707 L404.65182472282413 254.68539942375236 L410.9794529436658 255.39339396766547 L417.30708116450745 256.0988856658805 L423.63470938534914 256.8020257189495 L429.9623376061908 257.5029565907127 L436.2899658270324 258.201812232537 L442.61759404787415 258.8987183724242 L448.9452222687158 259.5937928503472 L455.2728504895574 260.28714598437824 L461.6004787103991 260.97888095505596 L467.92810693124073 261.6690941979607 L474.25573515208237 262.35787579663673 L480.583363372924 263.0453098698318 L486.91099159376574 263.73147494854777 L493.2386198146074 264.41644433964893 L499.566248035449 265.10028647378243 L505.8938762562907 265.78306523617744 L512.2215044771324 266.4648402795219 L518.549132697974 267.14566731861254 L524.8767609188156 267.8255984068451 L524.8767609188156 281.26010079229366 L518.549132697974 280.332487843936 L512.2215044771324 279.40577084643644 L505.8938762562907 278.48000185319074 L499.566248035449 277.55523657899556 L493.2386198146074 276.63153467653893 L486.91099159376574 275.7089600310499 L480.583363372924 274.7875810731757 L474.25573515208237 273.8674711097806 L467.92810693124073 272.9487086718665 L461.6004787103991 272.0313778781811 L455.2728504895574 271.11556881226863 L448.9452222687158 270.2013779097095 L442.61759404787415 269.2889083510423 L436.2899658270324 268.3782704543394 L429.9623376061908 267.46958205957344 L423.63470938534914 266.5629688947465 L417.30708116450745 265.65856491122537 L410.9794529436658 264.7565125728502 L404.65182472282413 263.85696308017316 L398.32419650198244 262.96007650770827 L391.9965682811408 262.0660218284615 L385.6689400602992 261.1749767964366 L379.34131183945755 260.2871276545721 L373.0136836186158 

## 4. Memory Pressure Stability

Lookup time measured at baseline and under 100 MB live allocation.
Threshold: degradation < 10× baseline.

In [6]:
val t4Reflect = load(reflectDir, "test4_memory_pressure.csv", "reflect")
val t4Serialize = load(serializeDir, "test4_memory_pressure.csv", "serialize")

data class T4Summary(val cache: String, val baselineNs: Long, val pressureNs: Long, val degradation: Double)

fun t4Summary(rows: List<Map<String, String>>, cache: String): T4Summary {
    val r = rows.first()
    return T4Summary(
        cache = cache,
        baselineNs = r["test4_baseline_time_ns"]!!.toLongL(),
        pressureNs = r["test4_pressure_time_ns"]!!.toLongL(),
        degradation = r["test4_degradation_ratio"]!!.toDoubleL()
    )
}

val t4R = t4Summary(t4Reflect, "reflect")
val t4S = t4Summary(t4Serialize, "serialize")

println("Test 4 — Memory Pressure:")
println(
    "  reflect:   baseline=%,d ns  pressure=%,d ns  degradation=%.2fx".format(
        t4R.baselineNs,
        t4R.pressureNs,
        t4R.degradation
    )
)
println(
    "  serialize: baseline=%,d ns  pressure=%,d ns  degradation=%.2fx".format(
        t4S.baselineNs,
        t4S.pressureNs,
        t4S.degradation
    )
)

val t4Caches = listOf("reflect", "reflect", "serialize", "serialize")
val t4Condition = listOf("baseline", "pressure", "baseline", "pressure")
val t4Times = listOf(t4R.baselineNs, t4R.pressureNs, t4S.baselineNs, t4S.pressureNs).map { it.toDouble() }

val t4Data = mapOf("cache" to t4Caches, "condition" to t4Condition, "time_ns" to t4Times)

letsPlot(t4Data) +
        geomBar(stat = Stat.identity, position = positionDodge(0.9), alpha = 0.85) {
            x = "cache"; y = "time_ns"; fill = "condition"
        } +
        labs(
            title = "Lookup time: baseline vs 100 MB memory pressure  (ns per op)",
            x = "Cache", y = "Time per op (ns)", fill = "Condition"
        ) +
        ggsize(620, 380)

Test 4 — Memory Pressure:
  reflect:   baseline=36 ns  pressure=37 ns  degradation=1,03x
  serialize: baseline=36 ns  pressure=34 ns  degradation=0,94x


reflect 
 
 
 
 
 
 
 
 
 serialize 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 
 
 Lookup time: baseline vs 100 MB memory pressure (ns per op) 
 
 
 
 
 Time per op (ns) 
 
 
 
 
 Cache 
 
 
 
 
 
 
 
 
 Condition 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 baseline 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 pressure

## 5. Complexity Independence (Tiny / Small / Large)

Lookup time measured for types with 1, 2, and 10 fields.
20 samples per type, interleaved warmup. Median compared across types.
Threshold: max/min median spread < 15×.

In [7]:
val t5Reflect = load(reflectDir, "test5_type_complexity.csv", "reflect")
val t5Serialize = load(serializeDir, "test5_type_complexity.csv", "serialize")

val t5All = t5Reflect + t5Serialize

// Compute median time_ns per (cache, type_complexity)
fun median(vals: List<Long>): Long {
    val s = vals.sorted(); return if (s.size % 2 == 0) (s[s.size / 2 - 1] + s[s.size / 2]) / 2 else s[s.size / 2]
}

data class T5Point(val cache: String, val complexity: String, val fields: Int, val medianNs: Long)

val complexityOrder = listOf("Tiny", "Small", "Large")
val t5Points = listOf("reflect", "serialize").flatMap { cache ->
    val rows = t5All.filter { it["cache"] == cache }
    complexityOrder.map { cx ->
        val times = rows.filter { it["test5_type_complexity"] == cx }.map { it["time_ns"]!!.toLongL() }
        val fields = rows.filter { it["test5_type_complexity"] == cx }.firstOrNull()
            ?.get("test5_field_count")?.toIntOrNull() ?: 0
        T5Point(cache, cx, fields, if (times.isEmpty()) 0L else median(times))
    }
}

println("Test 5 — Complexity Independence (median ns per op):")
t5Points.forEach {
    println(
        "  %-12s  %-6s (%2d fields)  %,d ns".format(
            it.cache,
            it.complexity,
            it.fields,
            it.medianNs
        )
    )
}

val t5Data = mapOf(
    "cache" to t5Points.map { it.cache },
    "complexity" to t5Points.map { "${it.complexity}\n(${it.fields} fields)" },
    "median_ns" to t5Points.map { it.medianNs.toDouble() }
)
val t5XOrder = complexityOrder.map { cx ->
    t5Points.first { it.complexity == cx }.let { "${it.complexity}\n(${it.fields} fields)" }
}

letsPlot(t5Data) +
        geomBar(stat = Stat.identity, position = positionDodge(0.9), alpha = 0.85) {
            x = "complexity"; y = "median_ns"; fill = "cache"
        } +
        scaleXDiscrete(limits = t5XOrder) +
        labs(
            title = "Median lookup time by type complexity  (ns per op, 20 samples)",
            x = "Type", y = "Median time per op (ns)", fill = "Cache"
        ) +
        ggsize(650, 380)

Test 5 — Complexity Independence (median ns per op):
  reflect       Tiny   ( 1 fields)  36 ns
  reflect       Small  ( 2 fields)  18 ns
  reflect       Large  (10 fields)  18 ns
  serialize     Tiny   ( 1 fields)  41 ns
  serialize     Small  ( 2 fields)  27 ns
  serialize     Large  (10 fields)  27 ns


Tiny 
 
 
 (1 fields) 
 
 
 
 
 
 
 
 
 Small 
 
 
 (2 fields) 
 
 
 
 
 
 
 
 
 Large 
 
 
 (10 fields) 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 
 
 Median lookup time by type complexity (ns per op, 20 samples) 
 
 
 
 
 Median time per op (ns) 
 
 
 
 
 Type 
 
 
 
 
 
 
 
 
 Cache 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 reflect 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 serialize

## Summary

In [8]:
println("%-40s  %18s  %18s".format("Metric", "reflect", "serialize"))
println("-".repeat(80))
println(
    "%-40s  %,16d ns  %,16d ns".format(
        "Test 1: cache lookup (10k ops)",
    t1Points.find { it.label.contains("reflect") }!!.timeNs,
    t1Points.find { it.label.contains("serialize") }!!.timeNs
)
)
println("%-40s  %,16d ns  %,16d ns".format("Test 2: p50 (median per-op)", t2R.p50, t2S.p50))
println("%-40s  %16.2f x  %16.2f x".format("Test 2: P90/P10 spread", t2R.spread, t2S.spread))
println("%-40s  %16.6f    %16.6f  ".format("Test 3: regression slope (ns/op)", t3R.slope, t3S.slope))
println("%-40s  %16.2f x  %16.2f x".format("Test 4: memory pressure degradation", t4R.degradation, t4S.degradation))
val t5SpreadR = t5Points.filter { it.cache == "reflect" }
    .let { pts -> pts.maxOf { it.medianNs }.toDouble() / pts.minOf { it.medianNs }.coerceAtLeast(1L) }
val t5SpreadS = t5Points.filter { it.cache == "serialize" }
    .let { pts -> pts.maxOf { it.medianNs }.toDouble() / pts.minOf { it.medianNs }.coerceAtLeast(1L) }
println("%-40s  %16.2f x  %16.2f x".format("Test 5: complexity spread (max/min)", t5SpreadR, t5SpreadS))
println("-".repeat(80))
println("\nAll tests passed for both caches: O(1) lookup confirmed.")

Metric                                               reflect           serialize
--------------------------------------------------------------------------------
Test 1: cache lookup (10k ops)                          41 ns               144 ns
Test 2: p50 (median per-op)                             35 ns                34 ns
Test 2: P90/P10 spread                                1,12 x              1,12 x
Test 3: regression slope (ns/op)                 -0,000139           -0,000349  
Test 4: memory pressure degradation                   1,03 x              0,94 x
Test 5: complexity spread (max/min)                   2,00 x              1,52 x
--------------------------------------------------------------------------------

All tests passed for both caches: O(1) lookup confirmed.
